# Lending Club loan selection: one-click rerun

**Capstone, AI for Finance (Idea C).** This notebook reruns the project from the raw data.
In Colab choose **Runtime > Run all**. Each step prints what it did.

It covers **every roadmap step that produces a result: 5 (clean data), 6-7 (models on 2013), 9 (2014 test), 10 (money), 11 (2015 robustness), 12 (interpretability), 13 (final tables and figures)** and the presentation-only redraw of Figure 3. Total run time is about 10 to 20 minutes; the 2015 step needs roughly 5 GB of free memory (standard Colab is enough).

**How failures are handled.** Every command runs through `run()` below, which stops the notebook at the first error. Each stage is launched by `src.reproduce_all --stage`, which records its result. The last cell prints `REPRODUCED` only if **every stage of this run succeeded** on the pinned code revision and **every expected result file** was rewritten by this run and is byte-identical to the committed version. Unchanged committed files alone can never pass.

**About the one-time tests.** The 2014 test and 2015 run were opened once, after the methodology freeze; their logs are in the repository. This notebook does not repeat them as new tests: their `--reproduce` mode reruns the same frozen code into a separate folder, and the final check compares those tables byte for byte with the committed ones.

What you need:
* the raw file `accepted_2007_to_2018Q4.csv.gz` (Kaggle `wordsforthewise/lending-club`, about 375 MB), either through a Kaggle API key **or** a copy in your Google Drive;
* access to the repository (a Colab secret `GITHUB_TOKEN` if it is private).

## 0. Which code version is checked

`CODE_COMMIT` is the full 40-character commit of the code to reproduce. The notebook clones the repository, checks out exactly that commit and refuses to continue if the checkout is at any other revision or has modified files, also when a `lucplaisier` folder already exists.

**Version pin.** `CODE_COMMIT` is the submission's analysis commit `ba250b763c0bc3e75e1b0ee6b6fb39c80d0f21f0` (the merge of PR #16 on 27 Nov 2026), the version cited in the manuscript. This notebook lives in a later, launcher-only commit, because a file cannot contain its own commit identifier; it checks out and verifies the pinned commit. `ALLOW_UNPINNED = True` is only for testing another revision.

In [ ]:
import os, sys, subprocess, shutil

CODE_COMMIT = "ba250b763c0bc3e75e1b0ee6b6fb39c80d0f21f0"   # submission analysis commit (merge of PR #16, 27 Nov 2026)
ALLOW_UNPINNED = False    # True only for testing before the pin exists
REPO = "lucplaisier-coder/lucplaisier"

def run(cmd, **kw):
    """Run a command and stop the notebook if it fails (no silent continuation)."""
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, **kw)

if not CODE_COMMIT and not ALLOW_UNPINNED:
    raise RuntimeError("CODE_COMMIT is empty. Set the submission commit, or set ALLOW_UNPINNED = True to test.")
if CODE_COMMIT and len(CODE_COMMIT) != 40:
    raise RuntimeError("CODE_COMMIT must be the full 40-character commit SHA.")
print("Python used for every step:", sys.executable)

## 1. Get the code
If the repository is private, add a Colab secret called `GITHUB_TOKEN` (key icon on the left) with read access. If it is public, nothing is needed.

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("lucplaisier"):
    run(["git", "clone", "--quiet", url, "lucplaisier"])
    if CODE_COMMIT:
        run(["git", "-C", "lucplaisier", "checkout", "--quiet", "--detach", CODE_COMMIT])
os.chdir("lucplaisier")
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"],
                       capture_output=True, text=True, check=True).stdout.strip()
if CODE_COMMIT and head != CODE_COMMIT:
    raise RuntimeError(f"Wrong code version: checkout is at {head}, expected {CODE_COMMIT}. "
                     "Delete the lucplaisier folder (or check out the right commit) and run again.")
if dirty:
    raise RuntimeError("The checkout has modified tracked files:\n" + dirty)
print("Code version:", head, "(pinned)" if CODE_COMMIT else "(UNPINNED test run)")
EXPECT = ["--expect-commit", CODE_COMMIT] if CODE_COMMIT else []

## 2. Install the pinned package versions
The exact versions are in `requirements.txt`. They are installed into the same Python that runs every step (`sys.executable`), so the steps use exactly these versions.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run([sys.executable, "-c", "import sklearn, pandas, numpy, matplotlib; "
     "print('scikit-learn', sklearn.__version__, '| pandas', pandas.__version__, "
     "'| numpy', numpy.__version__, '| matplotlib', matplotlib.__version__)"])

## 3. Get the raw data (choose ONE option)
* **Option A, Google Drive:** put the file anywhere in your Drive and set `DRIVE_PATH` below.
* **Option B, Kaggle API:** add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com > Settings > API), and leave `DRIVE_PATH` empty.

Step 5 checks the file's SHA-256 fingerprint and stops if it is not the exact file used in the project.

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/accepted_2007_to_2018Q4.csv.gz"   # set to "" to use Kaggle

os.makedirs("data/raw", exist_ok=True)
target = "data/raw/accepted_2007_to_2018Q4.csv.gz"
if not os.path.exists(target):
    if DRIVE_PATH:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copyfile(DRIVE_PATH, target)
    else:
        from google.colab import userdata
        env = dict(os.environ, KAGGLE_USERNAME=userdata.get("KAGGLE_USERNAME"),
                   KAGGLE_KEY=userdata.get("KAGGLE_KEY"))
        run([sys.executable, "-c", "from src.raw_data import fetch_raw_file; fetch_raw_file('kaggle')"], env=env)
print(target, os.path.getsize(target), "bytes")

def stage(name):
    """Run one recorded stage of src.reproduce_all; stops the notebook if it fails."""
    run([sys.executable, "-m", "src.reproduce_all", "--stage", name, *EXPECT])

## 4. Step 5: build the clean dataset (about 1 to 3 minutes)
Checks the fingerprint, applies the sample filters, the approved 148-loan rate fix and the fixed input rules, and writes the clean files. The 2014 and 2015 outcomes go to a separate sealed file that the code only opens after the methodology freeze record.

In [ ]:
stage("step5")

## 5. Look at the step-5 summary tables

In [ ]:
import pandas as pd, json
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200)
for name in ["funnel_by_vintage", "resolution_by_vintage", "sample_by_vintage",
             "input_missing_share_by_vintage", "cash_identity_check_2010_2013"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/step5/{name}.csv"))
summary = json.load(open("outputs/step5/step5_run_summary.json"))
print("Raw file unchanged:", summary["raw_unchanged"], "| anything fitted:", summary["anything_fitted"])

## 6. Step 6: logistic regression + platform benchmark (a few seconds)
Fits the logit on 2010-2012 only, chooses C on 2013, runs the leakage alarm, and compares fund-all, the platform rule and the logit on 2013 at 80% within each grade.

In [ ]:
stage("step6")
for name in ["leakage_alarm_single_input_auc_2013", "logit_tuning_C_2013",
             "predictive_validation_2013", "financial_validation_2013_q80"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/step6/{name}.csv"))
display(Image("outputs/step6/reliability_logit_2013.png"))

## 7. Step 7: add histogram gradient boosting (about 20 seconds)
Fits HGB on 2010-2012 for the four fixed settings (early stopping off), chooses one on 2013, and compares all four approaches on 2013.

In [ ]:
stage("step7")
for name in ["hgb_tuning_2013", "predictive_validation_2013", "financial_validation_2013_q80"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/step7/{name}.csv"))
display(Image("outputs/step7/reliability_2013.png"))

## 8. Step 9: the 2014 test, reproduced (about 1 minute)
First the rehearsal on 2013 (as done before the freeze), then the frozen test code on 2014, written to `outputs/reproduce/step9/`.

In [ ]:
stage("step9_dryrun")
stage("step9_reproduce")
for name in ["financial_2014_q80", "bootstrap_contrasts_2014", "predictive_2014", "success_categories_2014"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/reproduce/step9/{name}.csv"))

## 9. Step 10: financial evaluation (a few seconds)
Selection-rate curve, sub-grade mix and the main money table, from the 2014 scores of the previous step.

In [ ]:
stage("step10")
display(Image("outputs/step10/output4_selection_rate_curve_2014.png"))

## 10. Step 11: the 2015 robustness run, reproduced (about 1 to 2 minutes, about 5 GB memory)

In [ ]:
stage("step11_dryrun")
stage("step11_reproduce")
for name in ["financial_2015_q80", "bootstrap_contrasts_2015", "grade_groups_2014_2015"]:
    print(f"\n=== {name} ===")
    display(pd.read_csv(f"outputs/reproduce/step11/{name}.csv"))

## 11. Step 12: what drives the models (about 1 minute)

In [ ]:
stage("step12")
display(pd.read_csv("outputs/step12/logit_odds_ratios_per_sd.csv"))

## 12. Step 13: the final tables and figures, and the Figure 3 redraw
Step 13 only collects saved results. The presentation stage redraws Figure 3 from the saved `outputs/final/figure3_data.csv` (no fitting).

In [ ]:
stage("step13")
stage("presentation")
display(Image("outputs/presentation/figure3_permutation_importance_2014.png"))
display(Markdown(open("docs/final_tables.md").read()))

## 13. Final check: is every committed result reproduced by this run?
Prints `REPRODUCED` only if every stage above succeeded in this run, the code is the pinned commit, and every expected result file is present, rewritten by this run and byte-identical to the committed version. Otherwise it lists each problem and the cell fails.

The last lines of this cell's output show the verdict in large type and as `VERDICT: REPRODUCED` or `VERDICT: NOT REPRODUCED`. Copy everything from this cell's output when you report the result. Rerunning only this cell is quick: it compares files and fits nothing.

In [ ]:
# Run the check and show its verdict here, in this cell (captured, so Colab cannot hide or reorder it).
check = subprocess.run([sys.executable, "-m", "src.reproduce_all", "--verify", *EXPECT],
                       capture_output=True, text=True)
print(check.stdout + check.stderr)
print("Run record (which stages ran, at which commit, with which packages):")
if os.path.exists("outputs/reproduce/run_record.json"):
    print(open("outputs/reproduce/run_record.json").read())
verdict = "REPRODUCED" if check.returncode == 0 else "NOT REPRODUCED"
display(Markdown(f"## VERDICT: {verdict}"))
print("=" * 60 + f"\nVERDICT: {verdict}\n" + "=" * 60)
if check.returncode != 0:
    raise RuntimeError("NOT REPRODUCED: see the problems listed above.")
